In [ ]:
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import r2_score

# =====================================================
# 1. CONFIGURATION (CHANGE ONLY THIS PART)
# =====================================================

DATA_PATH = r"C:\Users\Sam\Desktop\ML\task\Data.xlsx"
sheet_name = "Data_after_KFold_LSSVC"
CONFIG = {
    "optimizer": "SDOA",
    "population": 25,
    "iterations": 200,
    "cv": 5,
    "random_state": 42
}

# =====================================================
# 2. LOAD DATA
# =====================================================

df = pd.read_excel(DATA_PATH, sheet_name=sheet_name)
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

X = StandardScaler().fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=CONFIG["random_state"]
)

MODEL = {
    "name": "KRR",
    "builder": KernelRidge,
    "bounds": {
        "alpha": (0.1, 10.0, float),
        "gamma": (0.001, 1.0, float)
    }
}

# =====================================================
# 3. HELPER FUNCTIONS
# =====================================================

def bounds_to_arrays(bounds):
    lb, ub, cast = [], [], []
    for v in bounds.values():
        lb.append(v[0])
        ub.append(v[1])
        cast.append(v[2])
    return np.array(lb), np.array(ub), cast

def decode_params(vec, bounds, cast):
    decoded = {}
    for i, k in enumerate(bounds.keys()):
        decoded[k] = cast[i](vec[i])
    return decoded

def make_objective(model_builder, bounds, cast):
    def objective(vec):
        params = decode_params(vec, bounds, cast)
        model = model_builder(**params)

        score = cross_val_score(
            model,
            X_train,
            y_train,
            cv=CONFIG["cv"],
            scoring="r2",
            n_jobs=-1
        ).mean()

        return -score
    return objective

# =====================================================
# 4. SPOTTED DEER OPTIMIZATION ALGORITHM (SDOA)
# =====================================================

def SDOA(objective, lb, ub, N, T, cast):
    start = time.time()
    D = len(lb)

    # --- Initialize population ---
    pop = lb + np.random.rand(N, D) * (ub - lb)
    fit = np.array([objective(pop[i]) for i in range(N)])

    best_idx = np.argmin(fit)
    best = pop[best_idx].copy()
    best_fit = fit[best_idx]

    convergence = []
    log = []

    for t in range(T):
        r = 0.5 * (1 - t / T)  # control parameter for exploration vs exploitation

        mean_pop = np.mean(pop, axis=0)
        for i in range(N):
            # Spotted Deer movement inspired update
            prey = pop[np.random.randint(0, N)]
            candidate = pop[i] + r * np.random.randn(D) * (prey - pop[i]) + r * np.random.randn(D) * (mean_pop - pop[i])
            candidate = np.clip(candidate, lb, ub)
            f = objective(candidate)

            if f < fit[i]:
                pop[i] = candidate
                fit[i] = f
                if f < best_fit:
                    best, best_fit = candidate.copy(), f

        convergence.append(-best_fit)

        best_decoded = decode_params(best, MODEL["bounds"], cast)
        log.append([t + 1] + [best_decoded[k] for k in MODEL["bounds"].keys()] + [-best_fit])

        print(
            f"Iter {t+1:03d}, Best = "
            + ", ".join(f"{k}={v}" for k, v in best_decoded.items())
            + f", R2 = {-best_fit:.4f}"
        )

    runtime = time.time() - start
    return decode_params(best, MODEL["bounds"], cast), -best_fit, convergence, runtime, log

# =====================================================
# 5. RUN OPTIMIZATION
# =====================================================

lb, ub, cast = bounds_to_arrays(MODEL["bounds"])
objective = make_objective(MODEL["builder"], MODEL["bounds"], cast)

best, best_score, convergence, runtime, log = SDOA(
    objective, lb, ub, CONFIG["population"], CONFIG["iterations"], cast
)

# =====================================================
# 6. FINAL MODEL & REPORT
# =====================================================

best_params = best
final_model = MODEL["builder"](**best_params)
final_model.fit(X_train, y_train)

test_score = r2_score(y_test, final_model.predict(X_test))

# =====================================================
# 7. TABLES (EXCEL / PAPER READY)
# =====================================================

# Full iterations log
iter_cols = ["iteration"] + list(MODEL["bounds"].keys()) + ["best_cv_r2"]
iterations_df = pd.DataFrame(log, columns=iter_cols)

# Convergence per iteration
convergence_df = pd.DataFrame({"best_r2": convergence})

# Summary
summary_df = pd.DataFrame([{
    "Model": MODEL["name"],
    "Optimizer": CONFIG["optimizer"],
    "Runtime_sec": runtime,
    "Test_R2": test_score
}])

# Best hyperparameters table
best_params_df = pd.DataFrame({
    "parameters": list(best_params.keys()),
    "values": list(best_params.values())
})

# =========================
# PRINT RESULTS
# =========================
print("\n✅ Best Hyperparameters Table:")
print(best_params_df)

print("\n✅ Summary Table:")
print(summary_df)

print("\n✅ Iterations Log Preview:")
print(iterations_df.head(10))

print("\n✅ Convergence Preview:")
print(convergence_df.head(10))
